In [1]:
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install transformers accelerate trl peft bitsandbytes datasets
!pip install qwen-vl-utils

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-b4qq0fjy
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-b4qq0fjy
  Resolved https://github.com/unslothai/unsloth.git to commit 17e9714a98a361385511ae2a821899e934638876
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.6.8-py3-none-any.whl size=36249886 sha256=90d4f7a4ff5babbd954ceefde8af9ca640f047da6db0466b8d1402f45d2fd85f
  Stored in directory: /tmp/pip-ephem-wheel-cache-hblmwbu8/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.6.8
    Uninstalling unsloth-2026.6.8:
      Successfully uninstalled unsloth-2026.6.8


In [ ]:
# ==========================================
# FULL PIPELINE: TẢI ẢNH LOCAL & FINE-TUNE QWEN2-VL
# ==========================================

import os, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from PIL import Image
import requests
from io import BytesIO
from unsloth import FastVisionModel, is_bfloat16_supported
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.trainer import UnslothVisionDataCollator

# ────────────────────────────────────────────
# 1. KHỞI TẠO MODEL & LORA
# ────────────────────────────────────────────
print("⏳ Đang tải mô hình Qwen2-VL...")
model_id = "unsloth/Qwen2-VL-7B-Instruct"

model, tokenizer = FastVisionModel.from_pretrained(
    model_id, 
    load_in_4bit=True, 
    use_gradient_checkpointing="unsloth",
    max_seq_length=4096, # Mở khóa bộ nhớ phần cứng
)

# Chốt cứng giới hạn cho Tokenizer
tokenizer.model_max_length = 4096 

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision=False, finetune_language=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)
print("✅ Mô hình đã sẵn sàng!")

# ────────────────────────────────────────────
# 2. CẤU HÌNH ĐƯỜNG DẪN THƯ MỤC
# ────────────────────────────────────────────
DATASET_PATH  = "/kaggle/input/datasets/hhty4719/qlora-dataset/qlora_dataset.jsonl" # SỬA LẠI ĐƯỜNG DẪN NÀY NẾU CẦN
IMG_CACHE_DIR = "/kaggle/working/img_cache"
os.makedirs(IMG_CACHE_DIR, exist_ok=True)

# Tạo một bức ảnh xám dự phòng (Phòng trường hợp link ảnh gốc bị chết)
DUMMY_PATH = os.path.join(IMG_CACHE_DIR, "dummy.jpg")
if not os.path.exists(DUMMY_PATH):
    Image.new("RGB", (224, 224), color=(200, 200, 200)).save(DUMMY_PATH)

# ────────────────────────────────────────────
# 3. TẢI ẢNH ĐA LUỒNG TỪ INTERNET VỀ Ổ CỨNG
# ────────────────────────────────────────────
def url_to_filename(url):
    # Lấy 2 phần cuối của URL để làm tên file cho khỏi trùng
    p = url.split("/")
    return f"{p[-2]}_{p[-1]}"

def download_one(url):
    fp = os.path.join(IMG_CACHE_DIR, url_to_filename(url))
    if os.path.exists(fp): return fp
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        r.raise_for_status()
        Image.open(BytesIO(r.content)).convert("RGB").save(fp)
        return fp
    except Exception:
        return None

print("⏳ Đang đọc JSONL gốc...")
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    raw_samples = [json.loads(line) for line in f]

all_urls = list({u for s in raw_samples for u in s["images"]})
cached_count = len([f for f in os.listdir(IMG_CACHE_DIR) if f != "dummy.jpg"])

if cached_count < len(all_urls) * 0.95:
    print(f"📥 Bắt đầu tải {len(all_urls)} ảnh về ổ cứng (Local Cache)...")
    with ThreadPoolExecutor(max_workers=32) as ex:
        futs = {ex.submit(download_one, u): u for u in all_urls}
        done = fail = 0
        for f in as_completed(futs):
            done += 1
            if f.result() is None: fail += 1
            if done % 1000 == 0 or done == len(all_urls):
                print(f"   [{done}/{len(all_urls)}] Tải lỗi: {fail}")
else:
    print("✅ Ảnh đã nằm sẵn trong ổ cứng, bỏ qua bước download.")



In [ ]:
# ────────────────────────────────────────────
# 4. CHUYỂN ĐỔI, SPLIT DỮ LIỆU & LƯU TẬP TEST
# ────────────────────────────────────────────
print("⏳ Cấu trúc lại dữ liệu cho Qwen2-VL...")
READY_JSONL = "/kaggle/working/qwen2vl_ready_local.jsonl"

new_samples = []
for s in raw_samples:
    user_content = []
    for u in s["images"]:
        fp = os.path.join(IMG_CACHE_DIR, url_to_filename(u))
        img_uri = f"file://{fp}" if os.path.exists(fp) else f"file://{DUMMY_PATH}"
        user_content.append({"type": "image", "image": img_uri})
        
    user_content.append({"type": "text", "text": s["instruction"]})
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a professional car pricing AI expert."}]},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": [{"type": "text", "text": s["response"]}]}
    ]
    new_samples.append({"messages": messages})

with open(READY_JSONL, "w", encoding="utf-8") as f:
    for d in new_samples:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

# Tải lại dataset và SPLIT (90% Train - 10% Test)
full_dataset = load_dataset("json", data_files=READY_JSONL, split="train")
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=3407)

train_data = split_dataset["train"]
eval_data = split_dataset["test"]

# Lưu lại tập Test ra ổ cứng để ngày mai chạy Script Đánh giá
TEST_DATA_PATH = "/kaggle/working/test_dataset_for_eval.jsonl"
eval_data.to_json(TEST_DATA_PATH)

print(f"✅ Đã chia: {len(train_data)} Train | {len(eval_data)} Test")

In [ ]:
# ────────────────────────────────────────────
# 5. KHỞI TẠO TRAINER (BẢN ĐÁNH NHANH 1 EPOCH)
# ────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=8, 
    warmup_steps=10,
    
    # 🔥 CHỈ CHẠY ĐÚNG 1 VÒNG 🔥
    num_train_epochs=1, 
    
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="/kaggle/working/qlora_car_pricing_checkpoints",
    remove_unused_columns=False,
    dataset_text_field="messages",
    max_seq_length=4096, 
    dataset_kwargs={"skip_prepare_dataset": True},
    
    # 🔥 BỘ ĐÁNH GIÁ (Rút gọn) 🔥
    eval_strategy="epoch",  # Chỉ thi thử 1 lần duy nhất vào cuối vòng
    save_strategy="epoch",  # Lưu lại 1 Checkpoint duy nhất khi kết thúc
    # (Đã xóa load_best_model_at_end và metric_for_best_model vì không còn cần thiết)
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_data,
    eval_dataset=eval_data, 
    args=training_args,
    # 🔥 ĐÃ XÓA DÒNG CALLBACKS EARLY STOPPING Ở ĐÂY 🔥
)

print("=" * 50)
print("🚀 BẮT ĐẦU FINE-TUNING (1 EPOCH SIÊU TỐC) 🚀")
print("=" * 50)
trainer_stats = trainer.train()

In [ ]:
# ────────────────────────────────────────────
# 6. LƯU MÔ HÌNH TỐT NHẤT
# ────────────────────────────────────────────
FINAL_PATH = "/kaggle/working/qlora_car_pricing_adapter_final"
print("⏳ Đang lưu mô hình TỐT NHẤT...")
model.save_pretrained(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)
print("✅ HUẤN LUYỆN HOÀN TẤT!")

# Đánh giá mô hình 

In [2]:
# ==========================================
# SCRIPT ĐÁNH GIÁ CHUNG CUỘC (VÒNG CHẠY PHỤC HỒI)
# ==========================================
import json
import re
import os
import numpy as np
from tqdm import tqdm
from unsloth import FastVisionModel
import torch
from qwen_vl_utils import process_vision_info 

MODEL_PATH = "/kaggle/input/datasets/hhty4719/qwen-car-pricing-lora-v1/qlora_car_pricing_adapter_final" 
TEST_DATA_PATH = "/kaggle/input/datasets/hhty4719/qwen-car-pricing-lora-v1/test_dataset_for_eval.jsonl" 
IMG_CACHE_DIR = "/kaggle/input/datasets/hhty4719/qwen-car-pricing-lora-v1/img_cache" 

print("⏳ Đang thức tỉnh Chuyên gia AI Định giá...")
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_PATH, 
    load_in_4bit=True,
    max_seq_length=4096,
    device_map="cuda", 
)
FastVisionModel.for_inference(model) 
print("✅ AI đã vào vị trí!")

with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
    eval_samples = [json.loads(line) for line in f]
print(f"✅ Đã phát {len(eval_samples)} mẫu xe để kiểm tra.")

def extract_price(text):
    match = re.search(r'\$([0-9,]+)', text)
    if match:
        return float(match.group(1).replace(',', ''))
    return None

actual_prices = []
predicted_prices = []

print("⏳ Đang tiến hành định giá tự động hàng loạt...")
for idx, sample in enumerate(tqdm(eval_samples)):
    messages = sample["messages"]
    
    actual_price = extract_price(messages[-1]["content"][0]["text"])
    prompt_messages = messages[:-1] 
    
    clean_messages = []
    for msg in prompt_messages:
        if isinstance(msg.get("content"), list):
            clean_content = []
            for item in msg["content"]:
                if item.get("type") == "image":
                    filename = item["image"].split("/")[-1]
                    real_path = os.path.join(IMG_CACHE_DIR, filename)
                    clean_content.append({
                        "type": "image", 
                        "image": f"file://{real_path}" 
                    })
                elif item.get("type") == "text":
                    clean_content.append({
                        "type": "text", 
                        "text": item.get("text", "")
                    })
            clean_messages.append({"role": msg["role"], "content": clean_content})
        else:
            clean_messages.append(msg)

    text = tokenizer.apply_chat_template(
        clean_messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    image_inputs, video_inputs = process_vision_info(clean_messages)
    
    inputs = tokenizer(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=150, 
        use_cache=True, 
        temperature=0.1 
    )
    
    generated_ids = outputs[:, inputs.input_ids.shape[1]:]
    prediction_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    predicted_price = extract_price(prediction_text)
    
    if actual_price and predicted_price:
        actual_prices.append(actual_price)
        predicted_prices.append(predicted_price)
        
        if idx < 3:
            print(f"\n--- Xe thứ {idx+1} ---")
            print(f"🗣️ NGUYÊN VĂN AI NÓI: '{prediction_text}'")
            print(f"🤖 AI dự đoán : ${predicted_price:,.0f}")
            print(f"✅ Giá thực tế : ${actual_price:,.0f}")

actuals = np.array(actual_prices)
preds = np.array(predicted_prices)

mae = np.mean(np.abs(actuals - preds))
mape = np.mean(np.abs((actuals - preds) / actuals)) * 100

print("\n" + "="*50)
print("🏆 BÁO CÁO NĂNG LỰC ĐỊNH GIÁ XE 🏆")
print("="*50)
print(f"Tổng số xe đã test : {len(actuals)} chiếc")
print(f"Sai số tuyệt đối trung bình (MAE) : ± ${mae:,.0f} / xe")
print(f"Độ lệch phần trăm trung bình (MAPE): {mape:.2f}%")
print("="*50)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Đang thức tỉnh Chuyên gia AI Định giá...
==((====))==  Unsloth 2026.6.8: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

✅ AI đã vào vị trí!
✅ Đã phát 521 mẫu xe để kiểm tra.
⏳ Đang tiến hành định giá tự động hàng loạt...


  0%|          | 1/521 [00:29<4:12:18, 29.11s/it]


--- Xe thứ 1 ---
🗣️ NGUYÊN VĂN AI NÓI: 'Based on the overall assessment of the 2020 Kia SUV, with a current odometer reading of 65,872 miles and the condition shown in the images, a reasonable estimated price for this vehicle is $25,995.'
🤖 AI dự đoán : $25,995
✅ Giá thực tế : $23,817


...


--- Xe thứ 2 ---
🗣️ NGUYÊN VĂN AI NÓI: 'Based on the overall assessment of the 2024 Hyundai SUV, with a current odometer reading of 20,199 miles and the condition shown in the images, a reasonable estimated price for this vehicle is $29,995.'
🤖 AI dự đoán : $29,995
✅ Giá thực tế : $29,750


  1%|          | 3/521 [00:57<2:27:37, 17.10s/it]Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Xe thứ 3 ---
🗣️ NGUYÊN VĂN AI NÓI: 'Based on the overall assessment of the 2022 Tesla Sedan, with a current odometer reading of 40,716 miles and the condition shown in the images, a reasonable estimated price for this vehicle is $25,995.'
🤖 AI dự đoán : $25,995
✅ Giá thực tế : $28,395


100%|██████████| 521/521 [1:55:35<00:00, 13.31s/it]


🏆 BÁO CÁO NĂNG LỰC ĐỊNH GIÁ XE 🏆
Tổng số xe đã test : 521 chiếc
Sai số tuyệt đối trung bình (MAE) : ± $2,728 / xe
Độ lệch phần trăm trung bình (MAPE): 7.38%
